# A Simple RAG Demo Project

Before running the notebook, follow the setup instructions in [README.md](README.md).

In [164]:

# Run once when setting up the project in Colab
#!git clone https://github.com/raivisskadins/simple-rag-demo.git
#%cd simple-rag-demo
#!git pull

# Run once if packages are missing (required also for Colab)
#!pip install -r requirements.txt

# Run to pull latest changes from GitHub
#!git pull


In [165]:
# ==============================
# 1. Imports
# ==============================

import os
import sys
import numpy as np
import faiss

from pathlib import Path

from sentence_transformers import SentenceTransformer
from openai import AzureOpenAI
from dotenv import load_dotenv


In [166]:
# ==============================
# 2. Load environment variables
# ==============================

# Load environment variables
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import userdata
    os.environ["AZURE_OPENAI_API_KEY"] = userdata.get("AZURE_OPENAI_API_KEY")
    os.environ["AZURE_OPENAI_API_VERSION"] = userdata.get("AZURE_OPENAI_API_VERSION")
    os.environ["AZURE_OPENAI_ENDPOINT"] = userdata.get("AZURE_OPENAI_ENDPOINT")
    os.environ["AZURE_OPENAI_CHAT_DEPLOYMENT"] = userdata.get("AZURE_OPENAI_CHAT_DEPLOYMENT")
else:
    load_dotenv()

client = AzureOpenAI(
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT")
)

chat_deployment = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT")
embedding_model = SentenceTransformer("paraphrase-multilingual-mpnet-base-v2")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [167]:
# ==============================
# 3. Load knowledge base
# ==============================

chunks = []

KNOWLEDGE_PATH = Path("data/")

for file_path in KNOWLEDGE_PATH.iterdir():
    if file_path.is_file():
        with open(file_path, "r", encoding="utf-8") as f:
            text = f.read()
            chunks += [p.strip() for p in text.split("\n\n") if p.strip()]

print("Chunks:")
for c in chunks:
    print("-", c, "\n")


Chunks:
- Humans only use 2% of their brains, the Great Wall of China is visible from the Moon, drinking eight liters of water a day is essential for survival, and lightning never strikes the same place twice. 

- Mount Everest is the tallest mountain on Earth. Its peak is 8848 meters above sea level and it is part of the Himalayas. 

- The Amazon rainforest is the largest tropical rainforest in the world. It spans multiple countries in South America and contains immense biodiversity. 

- The Pacific Ocean is the largest ocean on Earth. It covers more than 30% of the planet’s surface. 

- What you CAN throw away in the brown bin:
- Fruit and vegetable leftovers
- Meat and fish waste
- Cereal products and bread (without packaging)
- Eggshells
- Coffee grounds and tea 
- Cheese, cottage cheese and other solid dairy products
- Plants and trees no more than half a metre high, small branches 

- What you CANNOT throw away in the brown bin
- Liquids, including soap-based liquids
- Soups, sau

In [168]:
# ==============================
# 4. Create embeddings
# ==============================

def get_embedding(text):
    emb = embedding_model.encode(text)
    emb = emb / np.linalg.norm(emb)
    return emb

chunk_embeddings = np.array(
    [get_embedding(chunk) for chunk in chunks]
).astype("float32")
embedding_dim = chunk_embeddings.shape[1]



# ==============================
# 5. Create FAISS index
# ==============================

index = faiss.IndexFlatIP(embedding_dim)
# remove previously added content
index.reset()
index.add(chunk_embeddings)

print("FAISS index size:", index.ntotal)


FAISS index size: 10


In [169]:
# ==============================
# 6. User question
# ==============================

#question = "Which ocean is the largest on Earth?"
# question = "Is Great Wall of China visible from the Moon?"
#question = "Can i throw batteries into the yellow bin?"

print("Enter your question: ")
question = input()

#print("Question:", question)


# ==============================
# 7. Embed the question
# ==============================

question_embedding = np.array(
    [get_embedding(question)]
).astype("float32")

# You can print out the embedding just in case you want to see how it looks
#print("Question Embedding:", question_embedding)



Enter your question: 


In [170]:
# ==============================
# 8. Retrieve top 2 chunks
# ==============================

k = 2

distances, indices = index.search(question_embedding, k)

retrieved_chunks = [chunks[i] for i in indices[0]]

print("Retrieved context:")
for c in retrieved_chunks:
    print("-", c)


Retrieved context:
- What you CAN throw away in the brown bin:
- Fruit and vegetable leftovers
- Meat and fish waste
- Cereal products and bread (without packaging)
- Eggshells
- Coffee grounds and tea 
- Cheese, cottage cheese and other solid dairy products
- Plants and trees no more than half a metre high, small branches
- What you CANNOT throw away in the brown bin
- Liquids, including soap-based liquids
- Soups, sauces and oils
- Any packaging, including bio-film/cellophane
- Stones and fertiliser for the garden
- Pet toilet litter and pet waste
- Cinders, barbecue charcoal, cigarettes and ashes
- Hay, manure and other materials that are used in agriculture
- Large branches
- Nappies and other hygiene items


In [171]:
# ==============================
# 9. Create RAG prompt
# ==============================

context = "\n\n".join(retrieved_chunks)

prompt = f"""
# Instructions
Answer the following question in one short response.
Use only the context provided, even if it contains false facts.
Say "There is no information available" if there is no answer in the context, but only after carefully analysing all of the context

# Context:
{context}

# Question:
{question}

# Answer:
"""

print("Prompt:", prompt)


Prompt: 
# Instructions
Answer the following question in one short response.
Use only the context provided, even if it contains false facts.
Say "There is no information available" if there is no answer in the context, but only after carefully analysing all of the context

# Context:
What you CAN throw away in the brown bin:
- Fruit and vegetable leftovers
- Meat and fish waste
- Cereal products and bread (without packaging)
- Eggshells
- Coffee grounds and tea 
- Cheese, cottage cheese and other solid dairy products
- Plants and trees no more than half a metre high, small branches

What you CANNOT throw away in the brown bin
- Liquids, including soap-based liquids
- Soups, sauces and oils
- Any packaging, including bio-film/cellophane
- Stones and fertiliser for the garden
- Pet toilet litter and pet waste
- Cinders, barbecue charcoal, cigarettes and ashes
- Hay, manure and other materials that are used in agriculture
- Large branches
- Nappies and other hygiene items

# Question:
Can

In [172]:
# ==============================
# 10. Call GPT-4o to generate the answer
# ==============================

response = client.chat.completions.create(
    model=chat_deployment,
    messages=[
        {"role": "user", "content": prompt}
    ],
    temperature=0
)

print("\nLLM Answer:")
print(response.choices[0].message.content)


LLM Answer:
Yes, you can throw away cheese into the brown trash bin.
